# OCR-DENSE preparation

Extract Russian OCR text for MWS candidate pages and save a page-level cache.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, subprocess, sys
from pathlib import Path

def pip(args):
    print('pip', ' '.join(args))
    subprocess.check_call([sys.executable, '-m', 'pip'] + args)

marker = Path('/content/drive/MyDrive/experiments_vbert/.deps_ocr_dense_v2')

if not marker.exists():
    pkgs = [
        'numpy==1.26.4', 'pandas==2.2.2', 'Pillow==11.3.0', 'tqdm==4.67.1', 'pyarrow==19.0.1', 'requests==2.32.4', 'paddlepaddle-gpu==2.6.2', 'paddleocr==2.7.3'
    ]
    pip(['install', '-q', '--no-cache-dir', '--force-reinstall'] + pkgs)
    marker.parent.mkdir(parents=True, exist_ok=True)
    marker.write_text('ok', encoding='utf-8')
    os.kill(os.getpid(), 9)

print('deps ok')


deps ok


In [3]:
from pathlib import Path
import json, time, zipfile
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from paddleocr import PaddleOCR

In [7]:
EXPERIMENT_ID = 'OCR-DENSE'
RUN_ID = time.strftime('%Y%m%d_%H%M%S')
OUT = Path('/content/drive/MyDrive/experiments_vbert/training_runs') / EXPERIMENT_ID / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
MWS_ZIP = Path('/content/drive/MyDrive/eval/mws_vision_retrieval_outputs.zip')
LOCAL = Path('/content/ocr_dense_mws')
LOCAL.mkdir(parents=True, exist_ok=True)
SMOKE_TEST = False

In [8]:
if not (LOCAL / 'pairs.jsonl').exists():
    with zipfile.ZipFile(MWS_ZIP) as z:
        z.extractall(LOCAL)
pairs = pd.read_json(LOCAL / 'pairs.jsonl', lines=True)
candidates = pairs[['image_id']].drop_duplicates().rename(columns={'image_id': 'candidate_id'})
candidates['image_path'] = candidates['candidate_id'].map(lambda x: str(LOCAL / 'images' / f'{x}.png'))
if SMOKE_TEST:
    candidates = candidates.head(8).reset_index(drop=True)
print('pages', len(candidates))

pages 373


In [9]:
ocr = PaddleOCR(lang='ru', use_angle_cls=False, show_log=False)
rows = []
for r in tqdm(candidates.itertuples(index=False), total=len(candidates), desc='ocr mws'):
    try:
        pred = ocr.ocr(np.array(Image.open(r.image_path).convert('RGB')), cls=False) or []
        texts = []
        for page in pred:
            for _, rec in page or []:
                text = rec[0] if isinstance(rec, (list, tuple)) else rec
                if str(text).strip():
                    texts.append(str(text).strip())
        ocr_text = ' '.join(texts)
    except Exception:
        ocr_text = ''
    rows.append(dict(candidate_id=r.candidate_id, image_path=r.image_path, ocr_text=ocr_text))
ocr_pages = pd.DataFrame(rows)
ocr_pages.to_parquet(OUT / 'mws_ocr_pages.parquet', index=False)
ocr_pages.to_csv(OUT / 'mws_ocr_pages.csv', index=False)
(OUT / 'training_config.json').write_text(json.dumps({'experiment_id': EXPERIMENT_ID, 'ocr_lang': 'ru'}, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved to', OUT)

ocr mws:   0%|          | 0/373 [00:00<?, ?it/s]

saved to /content/drive/MyDrive/experiments_vbert/training_runs/OCR-DENSE/20260512_031525
